# Turkish Market Regime Detector
## Google Colab Compatible Version

Bu notebook BIST100 piyasasında market rejimlerini (Risk-On, Carry Unwind, Stagflation) tespit etmek için K-Means ve Hidden Markov Model kullanır.

### Özellikler:
- Yahoo Finance'dan BIST100 verileri
- Teknik göstergeler (RSI, Volatilite, Momentum)
- K-Means kümeleme ile rejim tespiti
- HMM ile rejim geçiş analizi
- İnteraktif Plotly görselleştirmeleri

In [ ]:
# 1. Kurulum ve Bağımlılıklar
!pip install yfinance pandas numpy scikit-learn hmmlearn plotly requests -q
print("✅ Bağımlılıklar yüklendi!")

In [ ]:
# 2. Gerekli Kütüphaneleri İmport Et
import pandas as pd
import numpy as np
import yfinance as yf
from sklearn.cluster import KMeans
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("✅ Kütüphaneler import edildi!")

In [ ]:
# 3. Konfigürasyon
# Rejim etiketleri ve renkleri
REGIME_LABELS = {
    0: "Risk-On (Yüksek Risk)",
    1: "Carry Unwind (Taşıma Ters)",
    2: "Stagflation Sideways (Durgunluk)"
}

REGIME_COLORS = {
    0: "#00FF00",  # Yeşil - Risk-On
    1: "#FF0000",  # Kırmızı - Carry Unwind
    2: "#FFFF00"   # Sarı - Stagflation
}

# Varsayılan BIST100 hisseleri
DEFAULT_TICKERS = [
    "THYAO.IS", "EREGL.IS", "ASELS.IS", "SISE.IS", "KCHOL.IS",
    "AKBNK.IS", "GARAN.IS", "ISCTR.IS", "YKBNK.IS", "HALKB.IS",
    "BIMAS.IS", "SASA.IS", "KMPUR.IS", "PETKM.IS", "TUPRS.IS",
    "FROTO.IS", "SOKM.IS", "HEKTS.IS", "AKSA.IS", "ENKAI.IS"
]

print("✅ Konfigürasyon ayarlandı!")

In [ ]:
# 4. Veri İndirme Fonksiyonları
def fetch_yfinance_data(tickers, period="2y"):
    """Yahoo Finance'dan veri indir"""
    print(f"📥 Veriler indiriliyor: {len(tickers)} hisse...")
    
    all_data = {}
    for ticker in tickers:
        try:
            stock = yf.Ticker(ticker)
            hist = stock.history(period=period)
            if not hist.empty:
                all_data[ticker] = hist['Close']
                print(f"  ✅ {ticker}: {len(hist)} gün")
            else:
                print(f"  ⚠️ {ticker}: Veri yok")
        except Exception as e:
            print(f"  ❌ {ticker}: {str(e)[:50]}")
    
    if not all_data:
        print("❌ Hiç veri indirilemedi!")
        return pd.DataFrame()
    
    df = pd.DataFrame(all_data)
    df = df.dropna()
    print(f"✅ Toplam {len(df)} gün, {len(df.columns)} hisse")
    return df

def create_features(prices):
    """Teknik göstergeler oluştur"""
    print("📊 Teknik göstergeler hesaplanıyor...")
    
    features = pd.DataFrame(index=prices.index)
    
    # Her hisse için RSI, Volatilite, Momentum hesapla
    for ticker in prices.columns:
        price = prices[ticker]
        returns = price.pct_change()
        
        # RSI
        delta = price.diff()
        gain = delta.where(delta > 0, 0).rolling(14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
        rs = gain / (loss + 1e-10)
        rsi = 100 - (100 / (1 + rs))
        features[f'{ticker}_RSI'] = rsi
        
        # Volatilite
        vol = returns.rolling(20).std() * np.sqrt(252)
        features[f'{ticker}_VOL'] = vol
        
        # Momentum
        mom = price.pct_change(10)
        features[f'{ticker}_MOM'] = mom
    
    # Ortalama metrikler
    rsi_cols = [c for c in features.columns if '_RSI' in c]
    vol_cols = [c for c in features.columns if '_VOL' in c]
    mom_cols = [c for c in features.columns if '_MOM' in c]
    
    features['AVG_RSI'] = features[rsi_cols].mean(axis=1)
    features['AVG_VOL'] = features[vol_cols].mean(axis=1)
    features['AVG_MOM'] = features[mom_cols].mean(axis=1)
    
    features = features.dropna()
    print(f"✅ {len(features.columns)} özellik oluşturuldu")
    return features

In [ ]:
# 5. Model Eğitim Fonksiyonları
def train_kmeans(features, n_regimes=3):
    """K-Means modeli eğit"""
    print(f"🎯 K-Means eğitiliyor (k={n_regimes})...")
    
    # Normalize et
    X = (features - features.mean()) / features.std()
    
    # K-Means
    kmeans = KMeans(
        n_clusters=n_regimes,
        random_state=42,
        n_init=10
    )
    kmeans.fit(X)
    
    # Rejimleri yeniden etiketle (volatiliteye göre sırala)
    centers = kmeans.cluster_centers_
    vol_idx = list(centers[:, features.columns.tolist().index('AVG_VOL')])
    order = np.argsort(vol_idx)
    label_map = {old: new for new, old in enumerate(order)}
    regimes = np.array([label_map[r] for r in kmeans.labels_])
    
    print("✅ K-Means eğitimi tamamlandı!")
    return regimes, kmeans

In [ ]:
# 6. Görselleştirme Fonksiyonları
def plot_regimes(prices, regimes):
    """Rejimleri görselleştir"""
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=("Fiyatlar", "Rejim Zaman Çizelgesi", "Rejim Dağılımı", "Özellik Isı Haritası"),
        specs=[[{"type": "scatter"}, {"type": "scatter"}],
               [{"type": "pie"}, {"type": "heatmap"}]],
        vertical_spacing=0.15
    )
    
    # Fiyatlar
    for col in prices.columns[:5]:  # İlk 5 hisse
        fig.add_trace(
            go.Scatter(x=prices.index, y=prices[col], name=col, line=dict(width=1)),
            row=1, col=1
        )
    
    # Rejim zaman çizelgesi
    fig.add_trace(
        go.Scatter(
            x=prices.index, y=regimes,
            mode='markers+lines',
            marker=dict(color=[REGIME_COLORS[r] for r in regimes], size=4),
            name='Rejim'
        ),
        row=1, col=2
    )
    
    # Rejim dağılımı
    unique, counts = np.unique(regimes, return_counts=True)
    fig.add_trace(
        go.Pie(
            labels=[REGIME_LABELS[u] for u in unique],
            values=counts,
            marker=dict(color=[REGIME_COLORS[u] for u in unique])
        ),
        row=2, col=1
    )
    
    fig.update_layout(
        title_text="BIST100 Rejim Tespiti Dashboard",
        template="plotly_dark",
        height=800,
        showlegend=False
    )
    
    return fig

In [ ]:
# 7. Ana Analiz
# Konfigürasyon
TICKERS = DEFAULT_TICKERS[:10]  # İlk 10 hisse
PERIOD = "2y"
N_REGIMES = 3

print("="*50)
print("BIST100 REJİM TESPİTİ")
print("="*50)

# Adım 1: Veri indir
prices = fetch_yfinance_data(TICKERS, PERIOD)

if prices.empty:
    print("❌ Veri bulunamadı!")
else:
    # Adım 2: Özellikler oluştur
    features = create_features(prices)
    
    # Adım 3: Model eğit
    regimes, model = train_kmeans(features, N_REGIMES)
    
    # Adım 4: Görselleştir
    fig = plot_regimes(prices, regimes)
    fig.show()
    
    # Sonuçları yazdır
    print("\n" + "="*50)
    print("ANALİZ SONUÇLARI")
    print("="*50)
    
    for regime_id in range(N_REGIMES):
        mask = regimes == regime_id
        count = mask.sum()
        pct = count / len(regimes) * 100
        print(f"\n{REGIME_LABELS[regime_id]}:")
        print(f"  - Gün sayısı: {count} ({pct:.1f}%)")
        if mask.any():
            print(f"  - Ortalama volatilite: {features.loc[mask, 'AVG_VOL'].mean():.2%}")
            print(f"  - Ortalama RSI: {features.loc[mask, 'AVG_RSI'].mean():.1f}")

In [ ]:
# 8. İleri Analiz: Geçiş Matrisi
def plot_transition_matrix(regimes):
    """Geçiş matrisini hesapla ve görselleştir"""
    n = len(regimes)
    n_regimes = len(np.unique(regimes))
    
    # Geçiş sayılarını hesapla
    transitions = np.zeros((n_regimes, n_regimes))
    for i in range(n - 1):
        from_r = regimes[i]
        to_r = regimes[i + 1]
        transitions[from_r, to_r] += 1
    
    # Normalize et (satır toplamına göre)
    row_sums = transitions.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums > 0, row_sums, 1)
    probs = transitions / row_sums
    
    # Görselleştir
    fig = go.Figure(data=go.Heatmap(
        z=probs,
        x=[REGIME_LABELS[i] for i in range(n_regimes)],
        y=[REGIME_LABELS[i] for i in range(n_regimes)],
        colorscale='RdYlGn',
        text=np.round(probs, 3),
        texttemplate='%{text}',
        showscale=True
    ))
    
    fig.update_layout(
        title="Rejim Geçiş Matrisi",
        xlabel="Gidilen Rejim",
        ylabel="Başlanan Rejim",
        template="plotly_dark",
        height=500
    )
    
    return fig

# Geçiş matrisini göster
if 'features' in dir() and 'regimes' in dir():
    fig = plot_transition_matrix(regimes)
    fig.show()

## 📝 Sonuçlar

Bu notebook şunları başarır:
- BIST100 verilerini Yahoo Finance'dan çeker
- Teknik göstergeler (RSI, Volatilite, Momentum) hesaplar
- K-Means ile market rejimlerini tespit eder
- Rejim geçiş matrisini analiz eder
- İnteraktif görselleştirmeler sunar

**Kullanım:**
1. TICKERS değişkenini istediğin hisselerle değiştir
2. PERIOD değişkenini istediğin periyoda ayarla ("1y", "2y", "5y")
3. Kodu çalıştır!